In [1]:
!pip install langchain langchain-core langchain-community langchain-experimental ptdantic

ERROR: Could not find a version that satisfies the requirement ptdantic (from versions: none)
ERROR: No matching distribution found for ptdantic


# Custom Tools

In [2]:
from langchain_core.tools import tool

In [3]:
#step 1 - Create a function
def multiply(a, b):
  """Multiply two numbers"""
  return a*b

In [5]:
#step 2 - add type hints
def multiply(a: float, b: float) -> float:
  """Multiply two numbers"""
  return a*b

In [6]:
#step 3 - add tool decorator

@tool
def multiply(a: int, b:int) -> int:
  """Mutiply two nubers"""
  return a*b

In [7]:
result = multiply.invoke({"a":3, "b":5})

In [8]:
print(result)

15


In [9]:
# you can also check like
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Mutiply two nubers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [10]:
# what LLM sees
print(multiply.args_schema.model_json_schema)

<bound method BaseModel.model_json_schema of <class 'langchain_core.utils.pydantic.multiply'>>


# Method 2 - Using StructuredTool

In [11]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [12]:
# schema
class multiplyInput(BaseModel):
  a: int = Field(required=True, description="first number")
  b: int = Field(required=True, description="second number")

/tmp/ipykernel_2081/295312491.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required=True, description="first number")
/tmp/ipykernel_2081/295312491.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description="second number")


In [13]:
def multiply_func(a: int, b:int)->int:
  return a*b
# since in the pydatic model you have already defined the integer type yu dont need to type hint in the function

In [14]:
#tool object
multiply_tool = StructuredTool.from_function(
    func=multiply_func, #function we created
    name="multiply",
    description="Multiplying two numbers",
    args_schema=multiplyInput  #pydantic model

)

In [15]:
# invoking the tool
result = multiply_tool.invoke({"a":4,"b":5})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

20
multiply
Multiplying two numbers


# Method-3 BaseTool Class


In [16]:
from langchain_core.tools import BaseTool
from typing import Type

In [17]:
# arg schema using pydantic
class MultiplyInput(BaseModel):
  a : int = Field(required=True, description="first number")
  b : int = Field(required=True, description="second number")

/tmp/ipykernel_2081/10880533.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a : int = Field(required=True, description="first number")
/tmp/ipykernel_2081/10880533.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b : int = Field(required=True, description="second number")


In [18]:
# basetool class
class MultiplyTool(BaseTool):
  name: str = "multiply"
  description: str = "Multiply two numbers"

  args_schema: Type[BaseModel] = MultiplyInput

  def _run(self, a:int, b:int)-> int:
    return a*b

In [19]:
multiply_tool = MultiplyTool()

In [20]:
result = multiply_tool.invoke({"a":5, "b":7})

In [21]:
print(result)

35
